### Carga de datos

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Configurar las rutas
current_dir = os.getcwd()
base_dir = os.path.dirname(current_dir) 

# Ajusta los nombres de tus dos archivos CSV aquí:
ruta_tejafood = os.path.join(base_dir, 'data', 'raw', 'Ventas_Feb3.csv')
ruta_tejamarket = os.path.join(base_dir, 'data', 'raw', 'TF_en_TM_Feb.csv')
ruta_tejafoodEnero = os.path.join(base_dir, 'data', 'raw', 'Ventas_Enero2.csv')
ruta_tejamarketEnero = os.path.join(base_dir, 'data', 'raw', 'TF_en_TM_Enr.csv')
ruta_tejafoodDic = os.path.join(base_dir, 'data', 'raw', 'Ventas_Dic.csv')
ruta_tejamarketDic = os.path.join(base_dir, 'data', 'raw', 'TF_en_TM_Dic.csv')

# 2. Cargar ambos archivos
print("⏳ Cargando archivos...")
df_tf = pd.read_csv(ruta_tejafood, sep=';', encoding='utf-8-sig')
df_tm = pd.read_csv(ruta_tejamarket, sep=';', encoding='utf-8-sig')
df_tf_enero = pd.read_csv(ruta_tejafoodEnero, sep=';', encoding='utf-8-sig')
df_tm_enero = pd.read_csv(ruta_tejamarketEnero, sep=';',encoding='utf-8-sig')
df_tf_dic = pd.read_csv(ruta_tejafoodDic, sep=';', encoding='utf-8-sig')
df_tm_dic = pd.read_csv(ruta_tejamarketDic, sep=';', encoding='utf-8-sig')

# 3. Unirlos en el DataFrame principal 'df'
df = pd.concat([df_tf, df_tm, df_tf_enero, df_tm_enero, df_tf_dic, df_tm_dic], ignore_index=True)

# 4. Crear el Identificador de Local inmediatamente
df['Num_Caja'] = df['ID_Ticket'].astype(str).str.split('-').str[0]
df['Local_Venta'] = np.where(
    df['Num_Caja'].isin(['10', '21']), 
    'Local Teja Food', 
    'Supermercado Teja Market'
)
df = df.drop(columns=['Num_Caja']) # Limpiamos la columna auxiliar

print(f"✅ ¡Archivos unidos y filtrados! Total filas: {len(df)}")
print("\nDistribución por local:")
print(df['Local_Venta'].value_counts())

⏳ Cargando archivos...
✅ ¡Archivos unidos y filtrados! Total filas: 49813

Distribución por local:
Local_Venta
Local Teja Food             36189
Supermercado Teja Market    13624
Name: count, dtype: int64


### Limpieza y Transformación

In [2]:
# 1. Convertir Fecha a formato datetime para poder hacer análisis temporal
df['Fecha'] = pd.to_datetime(df['Fecha'])

# 2. Calcular el Porcentaje de Margen (%) 
# Usamos un pequeño truco para evitar error de división por cero si Venta_Total es 0
df['Pct_Margen'] = (df['Margen_Neto'] / df['Venta_Total_Neta']).fillna(0)

# 3. Identificar productos con "Costo Cero" (Data Quality)
# Esto nos servirá para saber qué tan confiable es nuestro margen actual
df['Costo_Cargado'] = df['Costo_Unitario_Neto'] > 0

# 4. Resumen rápido de lo que tenemos
print(f"--- Resumen Ejecutivo Enero 2026 ---")
print(f"Venta Total Neta: ${df['Venta_Total_Neta'].sum():,.0f}")
print(f"Venta Total Bruta: ${df['Venta_Total_Bruta'].sum():,.0f}")
print(f"Margen Neto Total: ${df['Margen_Neto'].sum():,.0f}")
print(f"Ticket Promedio: ${df.groupby('ID_Ticket')['Venta_Total_Neta'].sum().mean():,.0f}")
print(f"Cantidad de Tickets: {df['ID_Ticket'].nunique()}")

display(df.head(1))

--- Resumen Ejecutivo Enero 2026 ---
Venta Total Neta: $202,069,014
Venta Total Bruta: $240,462,079
Margen Neto Total: $85,247,078
Ticket Promedio: $5,634
Cantidad de Tickets: 35864


,Fecha,Hora,Dia_Semana,ID_Ticket,PLU,Producto,Departamento,Linea,Unidades,Venta_Total_Bruta,Venta_Total_Neta,Costo_Unitario_Neto,Margen_Neto,Local_Venta,Pct_Margen,Costo_Cargado
0,2026-02-01,9,Sunday,10-5040615,21079,CAFE CHICO NESCAFE,TIENDA TEJA FOOD,CAFE,1.0,1690,1420.17,612.61,807.56,Local Teja Food,0.568636,True


### Recategorización

In [3]:
# --- BLOQUE DE RECATEGORIZACIÓN (CORREGIDO) ---

# 1. Cargar el maestro
ruta_maestro = os.path.join(base_dir, 'data', 'raw', 'dep_lin_pro.csv') 
if os.path.exists(ruta_maestro):
    df_maestro = pd.read_csv(ruta_maestro)
else:
    # Ojo: Si lo creas desde df, asegúrate de incluir 'Linea'
    df_maestro = df[['Producto', 'Departamento', 'Linea']].drop_duplicates()

# 2. Definir Palabras Clave (Listas arregladas)
keywords_gohan = ['GOHAN']
keywords_empanadas = ['EMPANADA']
keywords_platos = ['WRAP SALUDABLE PROTEICO 350 GRS APROX','SANDWICH PAN DE CERVEZA','POKE','HAMBURGUESA PAN DE CERVEZA 370 GRS APROX','ENSALADA','PIZZA','PASTA','PLATO','PASTEL DE CHOCLO', 'CHUPE', 'LASAÑA', 'QUICHE', 'TARTA', 'SOPA', 'CREMA DE', 'POROTOS', 'LENTEJAS']
# Unimos 'POSTRE' a la lista grande para no sobreescribir
keywords_postres = ['TORTA', 'KUCHEN', 'PIE DE', 'MOUSSE', 'TIRAMISU', 'SUSPIRO', 'CHEESECAKE', 'FLAN', 'JALEA', 'POSTRE']
keywords_confites = ['CHICLE', 'CARAMELO', 'GALLETA', 'ALFAJOR', 'BARRITA', 'GOMITA', 'PASTILLA', 'SNACK DULCE', 'M&M', 'ROLLS', 'SNICKERS', 'SUPER 8', 'NEGRI']
keywords_helados_env = ['HELADO','D´LIGHT','FRANUI', 'SAVORY', 'BRESLER', 'MAGNUM', 'CASSATA', 'PALETA', 'DANKY', 'CRAZY', 'MUSTY', 'MEGA']
keywords_tabaco = ['CONOS PAPEL FUMAR SOULBLIME','HEMP','PACK PIPA METALICA + REJILLA BULLDOG','PURITOS CAFE CREME RED CARTON X 10','CIGARRO', 'PALL MALL', 'LUCKY', 'KENT', 'DUNHILL', 'BELMONT', 'TABACO', 'OCB', 'PAPELILLO', 'FILTRO', 'ENCENDEDOR', 'VAP', 'VUSE']
keywords_marleycoffe = ['CAFE MEDIANO NESCAFE','CAFE CHICO NESCAFE','MARLEY']


# 3. La Función de Clasificación ROBUSTA
def asignar_nueva_categoria(nombre_prod, depto_original, linea_original):
    # Convertimos a string y mayúsculas para evitar errores si viene vacío
    nombre_upper = str(nombre_prod).upper()
    linea_upper = str(linea_original).upper()
    depto_upper = str(depto_original).upper()
    
    # --- NIVEL 1: REGLAS DE NEGOCIO ESPECÍFICAS (TEJA FOOD) ---
    # Si es del Teja Food, confiamos en su Línea (es más preciso que buscar palabras)
    if 'TEJA FOOD' in depto_upper:
        if 'POSTRE' in linea_upper: return 'POSTRES'
        # Si quieres separar los Gohan del Teja Food por línea también:
        if 'ACOMPAÑAMIENTOS' in linea_upper and 'GOHAN' in nombre_upper: return 'GOHAN'

    # --- NIVEL 2: GOHAN (Lógica Mixta) ---
    if any(k in nombre_upper for k in keywords_gohan): 
        # Si es externo O no dice 450 GRS -> Externo
        if 'EXTERNA' in depto_upper or '450 GRS' not in nombre_upper:
            return 'GOHAN EXTERNO'
        else:
            return 'GOHAN' # Interno

    # --- NIVEL 3: CATEGORÍAS POR PALABRAS CLAVE ---
    
    # Empanadas
    if any(k in nombre_upper for k in keywords_empanadas): return 'EMPANADAS'

    # Helados (Gelato vs Envasado)
    if 'GELATO' in nombre_upper:
        if 'DOBLE' in nombre_upper: return 'HELADOS - CONO'
        if 'SIMPLE' in nombre_upper: return 'HELADOS - CONO'
        return 'HELADOS - GELATO' # Por si falta el apellido
    
    if any(k in nombre_upper for k in keywords_helados_env): return 'HELADOS - ENVASADOS'

    # Tabaquería
    if any(k in nombre_upper for k in keywords_tabaco): return 'CIGARROS Y TABACO'

    # Postres (Por nombre)
    if any(k in nombre_upper for k in keywords_postres): return 'POSTRES'

    # Confites
    if any(k in nombre_upper for k in keywords_confites): return 'CONFITES'
    
    # # Bebestibles (Agregado de nuevo para no perderlos)
    # if any(k in nombre_upper for k in keywords_bebestibles): return 'BEBESTIBLES'

    # Platos Preparados
    if any(k in nombre_upper for k in keywords_platos): return 'PLATOS PREPARADOS'

    # Cafe Marley
    if any(k in nombre_upper for k in keywords_marleycoffe): return 'CAFE MARLEY'

    # --- NIVEL 4: DEFAULT ---
    # Si no cae en nada, devolvemos el Departamento Original
    return depto_original

# 4. Aplicar la función
print("Aplicando reglas de negocio corregidas...")
# Usamos axis=1 para pasar toda la fila a la función
df_maestro['Categoria_Jefe'] = df_maestro.apply(
    lambda x: asignar_nueva_categoria(x['Producto'], x['Departamento'], x['Linea']), axis=1
)

# 5. UNIR CON EL DATAFRAME PRINCIPAL
if 'Categoria_Jefe' in df.columns:
    df = df.drop(columns=['Categoria_Jefe'])

# Hacemos el merge. OJO: df debe tener la columna 'Producto' bien escrita
df = df.merge(df_maestro[['Producto', 'Categoria_Jefe']], on='Producto', how='left')

# Rellenar vacíos
df['Categoria_Jefe'] = df['Categoria_Jefe'].fillna(df['Departamento'])

print("\n✅ ¡Categorización actualizada correctamente!")
print("Ejemplo de distribución:")
print(df['Categoria_Jefe'].value_counts().head())
display(df.head(1))

Aplicando reglas de negocio corregidas...



✅ ¡Categorización actualizada correctamente!
Ejemplo de distribución:
Categoria_Jefe
CIGARROS Y TABACO        11421
PLATOS PREPARADOS         8836
EMPANADAS                 7801
BEBIDAS JUGOS Y AGUAS     5487
TEJA FOOD                 4829
Name: count, dtype: int64


,Fecha,Hora,Dia_Semana,ID_Ticket,PLU,Producto,Departamento,Linea,Unidades,Venta_Total_Bruta,Venta_Total_Neta,Costo_Unitario_Neto,Margen_Neto,Local_Venta,Pct_Margen,Costo_Cargado,Categoria_Jefe
0,2026-02-01,9,Sunday,10-5040615,21079,CAFE CHICO NESCAFE,TIENDA TEJA FOOD,CAFE,1.0,1690,1420.17,612.61,807.56,Local Teja Food,0.568636,True,CAFE MARLEY


In [4]:
# Asegurarnos de que Fecha es formato datetime
df['Fecha'] = pd.to_datetime(df['Fecha'])

# Crear una columna numérica para ordenar los días en Power BI (1=Lunes, 7=Domingo)
# dayofweek devuelve 0 para Lunes, así que le sumamos 1.
df['Dia_Semana_Num'] = df['Fecha'].dt.dayofweek + 1

# 3. TRADUCIR LOS DÍAS AL ESPAÑOL (Mapeo)
dias_espanol = {
    1: 'Lunes',
    2: 'Martes',
    3: 'Miércoles',
    4: 'Jueves',
    5: 'Viernes',
    6: 'Sábado',
    7: 'Domingo'
}
# Sobreescribimos la columna original de inglés al español
df['Dia_Semana'] = df['Dia_Semana_Num'].map(dias_espanol)

display(df[['Fecha', 'Dia_Semana', 'Dia_Semana_Num']].head(3))



,Fecha,Dia_Semana,Dia_Semana_Num
0,2026-02-01,Domingo,7
1,2026-02-01,Domingo,7
2,2026-02-01,Domingo,7


In [5]:
nombre_archivo_salida = os.path.join(base_dir, 'data', 'processed', 'MaestroPB_Ventastf.csv')

df.to_csv(nombre_archivo_salida, sep=';', index=False, encoding='utf-8-sig', decimal = ',')

### Pruebas

In [10]:
import pandas as pd

# 1. Cargar tu nuevo maestro de ventas (Asegúrate de poner el nombre correcto de tu archivo)
# Asumo que está separado por punto y coma (;) según la muestra que me diste antes

ruta_platos = os.path.join(base_dir, 'data', 'raw', 'Dim_Produccion_Limpia.csv')

# 2. Cargar la dimensión de producción que limpiamos antes
df_cuartos = pd.read_csv(ruta_platos, sep=';')

# 3. TRUCO DE ANALISTA: Asegurarnos de que el PLU sea texto (string) en ambas tablas 
# para que el cruce sea perfecto y no falle por decimales
df['PLU'] = df['PLU'].astype(str)
df_cuartos['PLU'] = df_cuartos['PLU'].astype(str)

# 4. Hacer el cruce (JOIN) entre ventas y producción
# Usamos 'inner' para que solo nos deje los productos que SÍ son de producción propia
df_cruce = pd.merge(df, df_cuartos, on='PLU', how='inner')

# 5. Calcular la cantidad de platos vendidos por Cuarto
resumen_cuartos = df_cruce.groupby('Tipo_Cuarto')['Unidades'].sum().reset_index()

# 6. Mostrar el resultado
print("🍽️ TOTAL DE PLATOS VENDIDOS POR CUARTO 🍽️")
display(resumen_cuartos)

# Opcional: Si quieres ver el detalle exacto de CADA preparación
detalle_preparaciones = df_cruce.groupby(['Tipo_Cuarto', 'Producto_y'])['Unidades'].sum().reset_index()
detalle_preparaciones = detalle_preparaciones.sort_values(by=['Tipo_Cuarto', 'Unidades'], ascending=[True, False])
print("\n📋 DETALLE POR PREPARACIÓN:")
display(detalle_preparaciones)

🍽️ TOTAL DE PLATOS VENDIDOS POR CUARTO 🍽️


,Tipo_Cuarto,Unidades
0,Cuarto Caliente,8391.256
1,Cuarto Frío,8312.000



📋 DETALLE POR PREPARACIÓN:


,Tipo_Cuarto,Producto_y,Unidades
26,Cuarto Caliente,POLLO ASADO,1816.256
10,Cuarto Caliente,LASAÑA BOLOÑESA,729.000
7,Cuarto Caliente,FIDEOS THAI CON CAMARONES,643.000
18,Cuarto Caliente,PASTEL DE CHOCLO,633.000
22,Cuarto Caliente,PASTEL DE CHOCLO MEDIANO,586.000
...,...,...,...
68,Cuarto Frío,QUICHE DE SALMON,14.000
81,Cuarto Frío,WRAP FALAFEL Y HUMMUS,13.000
70,Cuarto Frío,SANDWICH PAN DE CERVEZA JAMON SERRANO,5.000
77,Cuarto Frío,VASO KIWI,3.000
